####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 96d712fd-095a-4ace-93e8-8243dc7a3895
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 96d712fd-095a-4ace-93e8-8243dc7a3895 to get into ready status...
Session 96d712fd-095a-4ace-93e8-8243dc7a3895 

### Cast Table


In [2]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='glue_project', table_name='cast')
dyf.printSchema()

root
|-- cast: string
|-- show_id: string


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [3]:
cast_df = dyf.toDF()
cast_df.show()

+-----------------+--------+
|             cast| show_id|
+-----------------+--------+
|  Anjan Srivastav|80157133|
|        Doona Bae|70248183|
|  Ketchup Eusebio|81010870|
|      Ezra Miller|80038159|
| Charlie Hardwick|60003378|
|   Luci Christian|70204989|
|     Martin Starr|80106438|
|        Ragi Jani|80208413|
|      Sana Shaikh|70171835|
|   Miguel Pizarro|70205714|
| Michael Rapaport|70019506|
|     Ellen Barkin|70058024|
|    Soha Ali Khan|70106268|
|  Fred Tatasciore|81048914|
|     Glenn Fredly|81016323|
|   Ramya Krishnan|80203997|
|      Gustavo Vaz|80208298|
|Sebastián Aguirre|80106737|
|     Shia LaBeouf|70298926|
|   Loreto Peralta|81006825|
+-----------------+--------+
only showing top 20 rows

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
from pyspark.sql.functions import *

In [8]:
cast_df=cast_df.withColumn("First_name", split(col("cast")," ")[0])

In [10]:
cast_df=cast_df.withColumnRenamed("cast","cast_name")

In [11]:
cast_df.show()

+-----------------+--------+----------+
|        cast_name| show_id|First_name|
+-----------------+--------+----------+
|  Anjan Srivastav|80157133|     Anjan|
|        Doona Bae|70248183|     Doona|
|  Ketchup Eusebio|81010870|   Ketchup|
|      Ezra Miller|80038159|      Ezra|
| Charlie Hardwick|60003378|   Charlie|
|   Luci Christian|70204989|      Luci|
|     Martin Starr|80106438|    Martin|
|        Ragi Jani|80208413|      Ragi|
|      Sana Shaikh|70171835|      Sana|
|   Miguel Pizarro|70205714|    Miguel|
| Michael Rapaport|70019506|   Michael|
|     Ellen Barkin|70058024|     Ellen|
|    Soha Ali Khan|70106268|      Soha|
|  Fred Tatasciore|81048914|      Fred|
|     Glenn Fredly|81016323|     Glenn|
|   Ramya Krishnan|80203997|     Ramya|
|      Gustavo Vaz|80208298|   Gustavo|
|Sebastián Aguirre|80106737| Sebastián|
|     Shia LaBeouf|70298926|      Shia|
|   Loreto Peralta|81006825|    Loreto|
+-----------------+--------+----------+
only showing top 20 rows


In [12]:
# Writing the Tranformed file to the location
cast_df.write.mode("append")\
    .format("parquet")\
    .save("s3://project-glue-uk/silver_data/cast_silver/")

## Category Data

In [13]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='glue_project', table_name='category')
dyf.printSchema()

root
|-- listed_in: string
|-- show_id: string


In [15]:
cat_df=dyf.toDF()
cat_df.show()

+--------------------+--------+
|           listed_in| show_id|
+--------------------+--------+
|              Dramas|70100383|
|  Independent Movies|80152442|
|International TV ...|80993088|
|    Sci-Fi & Fantasy|60001761|
|International Movies|81035883|
|    Sci-Fi & Fantasy|80243541|
|            Comedies|81002215|
|  Action & Adventure|80177865|
|  Independent Movies|80987611|
|  Independent Movies|80200047|
|           TV Dramas|80158301|
|              Dramas|81024726|
|International TV ...|80108473|
|           TV Dramas|80148927|
|  Action & Adventure|80223052|
|International Movies|80100054|
|            Comedies|80028359|
|              Dramas|80244782|
| TV Sci-Fi & Fantasy|70242629|
|          Docuseries|70298341|
+--------------------+--------+
only showing top 20 rows


In [16]:
cat_df=cat_df.dropDuplicates()

In [17]:
cat_df.count()

13670


In [18]:
cat_df=cat_df.dropna(subset=["listed_in"])

In [23]:
cat_df=cat_df.withColumn("isTV", when(col("listed_in").like("%TV%"),True).otherwise(False))

In [25]:
cat_df.write.mode("append")\
    .format("parquet")\
    .save("s3://project-glue-uk/silver_data/category_silver/")

### Countries Data

In [34]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='glue_project', table_name='countries')
dyf.printSchema()

root
|-- country: string
|-- show_id: string


In [35]:
count_df=dyf.toDF()
count_df.show()

+-------------+--------+
|      country| show_id|
+-------------+--------+
|        Spain|81031939|
|  New Zealand|80085567|
|United States|70083532|
|United States|80133553|
|United States|80106136|
|      Germany|81030855|
|        China|80233085|
|       Russia|80192149|
|        India|70149579|
|United States|70125581|
|United States|60029366|
|        Japan|80996790|
|       Canada|70235760|
|        Japan|60029366|
|     Bulgaria|80218973|
|        India|70121660|
|    Hong Kong|80127645|
|       France|80166318|
|    Australia|81023011|
|United States|80201862|
+-------------+--------+
only showing top 20 rows


In [37]:
count_df=count_df.dropDuplicates()

In [40]:
count_df.count()

7179


In [39]:
count_df=count_df.dropna(subset=["country"])

In [41]:
count_df.write.mode("append")\
    .format("parquet")\
    .save("s3://project-glue-uk/silver_data/countries_silver/")

### Directors Data

In [42]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='glue_project', table_name='directors')
dyf.printSchema()

root
|-- director: string
|-- show_id: string


In [44]:
dir_df=dyf.toDF()
dir_df.show()

+--------------------+--------+
|            director| show_id|
+--------------------+--------+
|   Olivia M. Lamasan|81010865|
|       Jamie M. Dagg|80191294|
|      Yılmaz Erdoğan|70299745|
|      Anurag Kashyap|70113305|
|      Andrew Wessels|81228864|
|  Michael A. Nickles|81083786|
|     Benjamin Turner|80162144|
|     Juani Libonatti|81084225|
|       Sarah Moshman|80169548|
|        Samir Karnik|70012334|
|          Julia Hart|80105358|
|Ananth Narayan Ma...|80201825|
|       Mike Flanagan|80128722|
|       Eric Radomski|70295085|
|        Alex Lehmann|80104420|
|     Raj Kumar Gupta|70219528|
|  Christopher Storer|81078466|
|      Bruce Robinson|70120340|
|          Johnnie To|81030198|
|         Hakan Yonat|80134519|
+--------------------+--------+
only showing top 20 rows


In [45]:
dir_df.count()

4852


In [46]:
dir_df=dir_df.dropDuplicates()
dir_df=dir_df.dropna(subset=["director"])

In [47]:
dir_df.count()

4851


In [48]:
dir_df=dir_df.withColumn("Director_FirstName", split(col("director")," ")[0])


In [60]:
dir_df = dir_df.withColumn(
    "Director_LastName",
    split(col("director"), " ")[size(split(col("director"), " ")) - 1]
)

In [62]:
dir_df.write.mode("append")\
    .format("parquet")\
    .save("s3://project-glue-uk/silver_data/directors_silver/")